In [2]:
# Import pandas for data handling and table-style analysis
# Import matplotlib for creating charts and plots
# Import seaborn for attractive statistical visualizations

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Read the diabetes dataset from the CSV file into a DataFrame
# This creates a tabular object that we can inspect and analyze
df = pd.read_csv("diabetes_data/diabetic_data.csv")

In [4]:
# Show all column names to understand the available features and target variable
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [5]:
# Preview the first few rows to inspect the data structure and sample values
df.head(3)

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


In [6]:
# Check the unique values in the target column to understand the class labels
df['readmitted'].unique()

<StringArray>
['NO', '>30', '<30']
Length: 3, dtype: str

```bash
<30 → Patient was readmitted within 30 days of discharge.
🔹 Most critical (often hospitals are penalized if this happens).

>30 → Patient was readmitted, but only after 30 days.
🔹 Important, but less severe.

NO → Patient was not readmitted at all.
🔹 Majority class in this dataset.
```

In [7]:
# Define a helper function to compare class distributions before and after resampling
# It creates two side-by-side bar charts so the imbalance is easier to inspect
def plot_class_distribution(y_old, y_new, title):
    # Create a figure with two subplots placed side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    # Add a main title above both charts
    fig.suptitle(title, fontsize=16)

    # Plot the original target distribution on the left subplot
    sns.countplot(x=y_old, ax=ax1)
    # Label the left chart and rename category ticks for readability
    ax1.set_title('Original Distribution')
    ax1.set_xlabel('Readmission Category')
    ax1.set_ylabel('Count')
    ax1.set_xticks(ticks=[0, 1], labels=['NO / >30 days', '<30 days'])

    # Plot the resampled target distribution on the right subplot
    sns.countplot(x=y_new, ax=ax2)
    # Label the right chart the same way for easy comparison
    ax2.set_title('Resampled Distribution')
    ax2.set_xlabel('Readmission Category')
    ax2.set_ylabel('Count')
    ax2.set_xticks(ticks=[0, 1], labels=['NO / >30 days', '<30 days'])

    # Adjust layout so the title and charts fit neatly on the page
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    # Display the figure
    plt.show()

In [8]:
# Separate features from the target while removing identifier columns
# The target is the readmission outcome, and the dropped columns are not useful predictors
X = df.drop(columns=['readmitted', 'encounter_id', 'patient_nbr'], errors='ignore')
y = df['readmitted']

In [9]:
# Print a summary of the feature DataFrame to inspect dtypes and missing values
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 47 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   race                      101766 non-null  str  
 1   gender                    101766 non-null  str  
 2   age                       101766 non-null  str  
 3   weight                    101766 non-null  str  
 4   admission_type_id         101766 non-null  int64
 5   discharge_disposition_id  101766 non-null  int64
 6   admission_source_id       101766 non-null  int64
 7   time_in_hospital          101766 non-null  int64
 8   payer_code                101766 non-null  str  
 9   medical_specialty         101766 non-null  str  
 10  num_lab_procedures        101766 non-null  int64
 11  num_procedures            101766 non-null  int64
 12  num_medications           101766 non-null  int64
 13  number_outpatient         101766 non-null  int64
 14  number_emergency          10176

In [10]:
# Convert categorical columns into numeric dummy variables so resampling methods can use them
# pd.get_dummies creates one-hot encoded columns for each category
X = pd.get_dummies(X)

In [11]:
# Install imbalanced-learn if it is not already available in the environment
# This package provides oversampling and undersampling tools such as SMOTE and ADASYN
# !pip install imblearn

In [12]:
# Import resampling techniques that rebalance the class distribution
# RandomUnderSampler removes samples from the majority class
# RandomOverSampler, SMOTE, and ADASYN increase the minority class representation
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN

## What is Random Under-Sampling?

In an imbalanced dataset, you have a **majority class** (e.g., legitimate transactions) and a **minority class** (e.g., credit card fraud).

Random Under-Sampling balances the dataset by randomly selecting a subset of examples from the majority class and deleting them until the majority class matches the size of the minority class (or reaches a user-defined ratio).

## The Stats & Mechanics Behind It

The statistical core of RUS relies on **random uniform sampling without replacement** from the majority class population.

Let’s look at the mathematical effect on your data distributions:

- **Class Distribution Shift:** If your original dataset has a majority-to-minority ratio of $99:1$, RUS alters the joint distribution $P(X, Y)$ by changing the prior probability $P(Y)$. In a perfectly balanced undersampled set, $P(Y_{majority}) = P(Y_{minority}) = 0.5$.
- **Effect on Mean and Variance:** Because the samples from the majority class are selected completely at random, the sample mean ($\mu_{sample}$) of the undersampled majority class remains an unbiased estimator of the true population mean ($\mu_{population}$). However, because you are drastically reducing the sample size ($n$), the **variance of your estimators increases**.

According to the Central Limit Theorem, the standard error of the mean is:

$$
\text{SE} = \frac{\sigma}{\sqrt{n}}
$$

By shrinking $n$ (the number of majority samples), you increase the standard error, making your model's understanding of the majority class more volatile.

## Pros & Cons

### The Good (Pros)

- **Massive Speed & Efficiency:** By reducing the total number of training examples, you drastically decrease the computational time and memory required to train your model.
- **Simplifies Storage:** Less data means smaller storage footprints during the training phase.
- **No Artificial Data:** Unlike SMOTE or ADASYN, it does not create synthetic data points, ensuring your model trains only on real, observed data.

### The Bad (Cons)

- **Information Loss:** This is the biggest drawback. By deleting majority class examples, you throwing away potentially critical information, patterns, and boundary definitions that the model needs to learn.
- **High Variance:** Because you are training on a fraction of the majority data, the model's performance can vary wildly depending on *which* random samples were kept.

## When to Use It

RUS is highly effective under specific conditions:

1. **Massive Datasets:** If you have 10 million majority rows and 50,000 minority rows, you can afford to lose data. Training on 100,000 balanced rows is fast and often statistically sufficient.
2. **Severe Computational Constraints:** When you lack the RAM or time to run advanced algorithms on the full dataset.
3. **Clean, Distinct Classes:** If the majority class is highly homogeneous (meaning the data points are very similar to each other), removing data won't lose much unique information.

## Pitfalls to Avoid

- **The "Validation Leak" Pitfall:** Never apply under-sampling to your entire dataset before splitting. **You must split into Train/Test first, and only under-sample the training set.** Your test/validation set must remain imbalanced to reflect the real world.
- **Over-reducing:** You don't always have to balance to a perfect $50:50$ ratio. Sometimes, under-sampling a $99:1$ mix down to a $70:30$ or $80:20$ mix preserves enough majority information while still giving the minority class a fighting chance.
- **Ignoring Probability Calibration:** Because you artificially changed the class priors ($P(Y)$), the raw probability outputs of models like Logistic Regression or Random Forests will be skewed high for the minority class. If you need true probabilities, you must post-calibrate your model.

## Summary

**Random Under-Sampling** is a fast, computationally cheap baseline technique that balances data by randomly deleting majority class instances. While it drastically speeds up training, its fatal flaw is **information loss**, which can lead to high variance and a poorly generalized model if the majority class has complex patterns. It is best used as a quick starting point when dealing with massive datasets where data redundancy is high.

In [13]:
# 1. Random Under-sampling
# Create an undersampler that reduces the majority class so the minority class becomes half as large
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=4)
# The sampling_strategy parameter controls the target ratio between classes
# random_state ensures the resampling is reproducible